# AdaBoost, Gradient Boosting and XGBoost

Run from top to bottom. Quick mode is a patient subset, not final results. Notebook outputs are separate from production reports.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'features.py').exists()), None)
assert ROOT is not None, 'Open the notebook from the project repository'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.notebook_utils import *
QUICK = True  # False uses the entire eligible dataset; keep consistent across notebooks
OUT = ROOT/'reports'/('notebook_quick' if QUICK else 'notebook_full')
OUT.mkdir(parents=True, exist_ok=True)


Load identical splits

In [ ]:
raw, x, y, g, parts = load_cohort(ROOT, QUICK)
train, cal, tune, test = (parts[k] for k in ("train","calibration","threshold","test"))
X, Y, G = x.iloc[train], y.iloc[train], g.iloc[train]
display(split_table(y,g,parts))

Instantiate and train each classifier. Preprocessing is fitted only on training encounters.

In [ ]:
from src.models.boosting import models
candidates=models()
fitted={}; predictions={}; rows=[]
for name,est in candidates.items():
    model=make_model(X,est,quick=QUICK)
    model.fit(X,Y)
    prob=model.predict_proba(x.iloc[tune])[:,1]
    fitted[name]=model; predictions[name]=prob
    rows.append({'model':name,**scores(y.iloc[tune],prob)})
    print('Trained',name)
result=pd.DataFrame(rows); display(result); result.to_csv(OUT/'05_boosting_models.csv',index=False)

Compare validation metrics and curves

In [ ]:
result.set_index('model')[['roc_auc','pr_auc','recall','precision']].plot.bar(figsize=(10,4)); plt.tight_layout(); plt.show()
plot_curves(y.iloc[tune],predictions)